# k-meansとGMM+EM(演習)

解説資料は `research-handbook/machine-learning/clustering.md`。
このnotebookは**直接編集せず**、自分の卒研repoにコピーして使うこと(手順は `technical-handbook/colab/use-github-repo.md`)。

「---- ここから課題 ----」の区間を埋めながら上から順に実行する。
解答付き版は `research-handbook/notebooks/kmeans_gmm_solution.ipynb` にあるが、まず自力で取り組むこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def make_three_blobs(n=300, seed=0):
    rng = np.random.default_rng(seed)
    mus = np.array([[0, 0], [3.5, 2.0], [-2.5, 3.0]])
    X = np.vstack([rng.normal(mu, [0.7, 0.9], (n // 3, 2)) for mu in mus])
    z = np.repeat(np.arange(3), n // 3)
    return X, z, mus

X, z_true, mus_true = make_three_blobs()
plt.scatter(*X.T, c=z_true, cmap="viridis", s=10); plt.title("data"); plt.show()

## 課題1・2: k-means

歪み尺度 $J = \sum_n \|x_n - \mu_{k(n)}\|^2$ を、割り当て(E-step相当)と重心更新(M-step相当)の交互最適化で下げていきます。空クラスタの処理を忘れないこと(`clustering.md`)。

In [ ]:
def assign(X, centers):
    """課題1: 各点を最近傍の重心に割り当てる。戻り値は所属インデックス (N,)"""
    # ---- ここから課題1 ----
    # ヒント: ブロードキャストで全点-全重心の距離2乗 (N, K) を作り argmin
    return None
    # ---- ここまで ----

def update_centers(X, labels, K):
    """課題2: 各クラスタの平均で重心を更新する(空クラスタに注意)"""
    centers = np.zeros((K, X.shape[1]))
    # ---- ここから課題2 ----
    # 各 k について labels == k の点の平均を取る
    # 空クラスタの場合はランダムな点を重心にする等の処理を入れること


    # ---- ここまで ----
    return centers

def kmeans(X, K, n_iters=50, seed=0):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), K, replace=False)]   # データ点から初期化
    for _ in range(n_iters):
        labels = assign(X, centers)
        new_centers = update_centers(X, labels, K)
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
    distortion = ((X - centers[labels]) ** 2).sum()
    return labels, centers, distortion

labels, centers, J = kmeans(X, 3)
print(f"歪み尺度 J = {J:.1f}")
print("推定重心:"); print(np.sort(centers, axis=0))
print("真の中心:"); print(np.sort(mus_true.astype(float), axis=0))

## 課題3・4: 混合ガウスとEM

E-stepで**負担率**(各点が各クラスタに属する事後確率)

$$\gamma(z_{nk}) = \frac{\pi_k \mathcal{N}(x_n \mid \mu_k, \Sigma_k)}{\sum_j \pi_j \mathcal{N}(x_n \mid \mu_j, \Sigma_j)}$$

を計算し、M-stepで負担率重み付きの最尤推定を行います。**対数尤度が単調増加する**ことがEMの検算ポイントです。

In [ ]:
def gaussian_pdf(X, mu, cov):
    d = X.shape[1]
    diff = X - mu
    inv = np.linalg.inv(cov)
    norm = 1.0 / np.sqrt((2 * np.pi) ** d * np.linalg.det(cov))
    return norm * np.exp(-0.5 * np.einsum("ni,ij,nj->n", diff, inv, diff))

def e_step(X, pis, mus, covs):
    """課題3: E-step。負担率 γ(z_nk) = π_k N(x|μ_k,Σ_k) / Σ_j π_j N(x|μ_j,Σ_j)"""
    K = len(pis)
    # ---- ここから課題3 ----
    # 分子 (N, K) を作ってから行ごとに正規化する
    resp = None
    # ---- ここまで ----
    return resp

def m_step(X, resp):
    """課題4: M-step。負担率で重み付けした最尤推定"""
    N, K = resp.shape
    Nk = resp.sum(axis=0)                                 # (K,)
    # ---- ここから課題4 ----
    # clustering.md のM-stepの式: π_k = N_k/N、μ_k = Σ_n γ_nk x_n / N_k
    pis = None
    mus = None
    # ---- ここまで ----
    covs = []
    for k in range(K):
        diff = X - mus[k]
        covs.append((resp[:, k, None] * diff).T @ diff / Nk[k] + 1e-6 * np.eye(X.shape[1]))
    return pis, mus, np.array(covs)

def log_likelihood(X, pis, mus, covs):
    p = sum(pis[k] * gaussian_pdf(X, mus[k], covs[k]) for k in range(len(pis)))
    return np.log(p).sum()

def gmm_em(X, K, n_iters=100, seed=0):
    labels, centers, _ = kmeans(X, K, seed=seed)          # k-meansで初期化(定番)
    pis = np.bincount(labels, minlength=K) / len(X)
    mus = centers.copy()
    covs = np.array([np.cov(X[labels == k].T) + 1e-6 * np.eye(2) for k in range(K)])
    lls = []
    for _ in range(n_iters):
        resp = e_step(X, pis, mus, covs)
        pis, mus, covs = m_step(X, resp)
        lls.append(log_likelihood(X, pis, mus, covs))
        if len(lls) > 1 and abs(lls[-1] - lls[-2]) < 1e-8:
            break
    return pis, mus, covs, resp, lls

pis, mus, covs, resp, lls = gmm_em(X, 3)
print("混合係数:", pis)
print("対数尤度が単調増加しているか:", bool(np.all(np.diff(lls) >= -1e-8)))

plt.figure(figsize=(9, 3))
plt.subplot(1, 2, 1)
plt.plot(lls); plt.xlabel("EM iteration"); plt.ylabel("log likelihood")
plt.subplot(1, 2, 2)
plt.scatter(*X.T, c=resp @ np.arange(3), cmap="viridis", s=10)
plt.scatter(*mus.T, c="r", marker="x", s=100)
plt.tight_layout(); plt.show()

## ハード割り当てとソフト割り当て

k-meansは各点を1つのクラスタに完全に割り当てますが、GMMは確率で割り当てます。クラスタ境界付近の点で違いを確認します。

In [ ]:
# k-means(ハード割り当て)とGMM(ソフト割り当て)の違いをクラスタ境界付近で見る
boundary_pts = np.array([[1.7, 1.0], [0.0, 1.5], [-1.0, 1.5]])
resp_b = e_step(boundary_pts, pis, mus, covs)
for p, r in zip(boundary_pts, resp_b):
    print(f"x = {p} → 負担率 {np.round(r, 3)}")
# 境界付近では複数クラスタに確率が分散する(ソフト割り当て)

## 発展課題

1. K=2, 4, 5 で歪み尺度・BICを計算し、K の選び方(エルボー・BIC)を試す(`clustering.md`・`model-evaluation.md`)
2. 初期値のseedを変えて複数回実行し、局所最適に落ちる例を観察する(k-means++を実装してもよい)
3. 共分散を等方性 $\sigma^2 I$ に制約して $\sigma \to 0$ とすると負担率がハード割り当てに近づくことを数値で確認する(k-meansとの関係)

## まとめ

- k-means と GMM+EM はどちらも「割り当て→更新」の交互最適化。GMMは確率的な割り当て
- EMは対数尤度を単調に増やす。増えなかったら実装のバグ
- EMの枠組みはHMMのBaum-Welch(`hmm-kalman.md`)にそのまま一般化される